# analysis-anomaly-detection — Forecasting & anomaly scoring with Prophet

Fit a [Prophet](https://facebook.github.io/prophet/) model to a metric's recent
history, then score how far the latest points fall **outside** the model's
predicted uncertainty band. This is the interactive companion to the
`anomaly-scorer` service under `extra/anomaly-scorer/`, which runs exactly this
loop and re-exports the result as a Prometheus gauge.

What you'll do:

1. Range-query a Prometheus-compatible API (Prometheus, Mimir, GMP, Thanos) for a
   metric's history.
2. Fit Prophet on all-but-the-trailing window (trend + daily seasonality).
3. Predict the held-out tail and compare actual vs. the `[yhat_lower, yhat_upper]`
   band.
4. Turn the deviation into a 0..1 anomaly score and sketch how to export it.

Connection settings reuse the same env vars as `telemetry-metrics.ipynb`.

In [ ]:
import os

# --- Connection (same env vars as telemetry-metrics.ipynb) ---
PROM_URL    = os.environ.get('PROM_URL', os.environ.get('MIMIR_ADDRESS', 'http://prometheus:9090'))
PROM_TENANT = os.environ.get('MIMIR_TENANT_ID', '')   # blank for vanilla Prometheus
PROM_TOKEN  = os.environ.get('PROM_TOKEN', '')        # bearer token if any

# --- What to score ---
METRIC_QUERY = os.environ.get(
    'METRIC_QUERY',
    'container_memory_working_set_bytes{namespace="production"}',
)

# --- Model / windowing (mirror anomaly-scorer defaults) ---
LOOKBACK_DAYS     = float(os.environ.get('LOOKBACK_DAYS', '7'))    # history to train on
STEP              = os.environ.get('STEP', '5m')                   # query resolution
EVAL_POINTS       = int(os.environ.get('EVAL_POINTS', '12'))      # trailing points to score (12 x 5m = 1h)
INTERVAL_WIDTH    = float(os.environ.get('INTERVAL_WIDTH', '0.95'))
DAILY_SEASONALITY = os.environ.get('DAILY_SEASONALITY', 'true').lower() == 'true'

## Pull a metric's history

`query_range` returns one tidy `DataFrame[ds, y]` per series, keyed by its label
set — the column names (`ds`, `y`) are exactly what Prophet expects.

In [ ]:
import time, requests, pandas as pd

def _headers():
    h = {}
    if PROM_TENANT:
        h['X-Scope-OrgID'] = PROM_TENANT
    if PROM_TOKEN:
        h['Authorization'] = f'Bearer {PROM_TOKEN}'
    return h

def query_range(query, lookback_days=LOOKBACK_DAYS, step=STEP):
    end = int(time.time())
    start = end - int(lookback_days * 86400)
    r = requests.get(f'{PROM_URL}/api/v1/query_range',
                     params={'query': query, 'start': start, 'end': end, 'step': step},
                     headers=_headers(), timeout=120)
    r.raise_for_status()
    series = {}
    for s in r.json()['data']['result']:
        labels = ', '.join(f'{k}={v}' for k, v in sorted(s['metric'].items())) or 'value'
        df = pd.DataFrame(s['values'], columns=['ds', 'y'])
        df['ds'] = pd.to_datetime(df['ds'], unit='s')
        df['y'] = df['y'].astype(float)
        series[labels] = df
    return series

series = query_range(METRIC_QUERY)
print(f'{len(series)} series for: {METRIC_QUERY}')
assert series, 'No data — adjust METRIC_QUERY / PROM_URL (need ~%g days of history).' % LOOKBACK_DAYS

# Walk one series end-to-end; the anomaly-scorer container loops over all of them.
key, df = next(iter(series.items()))
print('series:', key, '| points:', len(df))
df.tail(3)

## Fit Prophet, hold out the tail

Train on everything except the last `EVAL_POINTS` samples, then predict that
held-out window. Prophet returns a point forecast `yhat` plus an uncertainty
band `[yhat_lower, yhat_upper]` sized by `INTERVAL_WIDTH`.

In [ ]:
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)  # quiet the fitter
from prophet import Prophet

train, tail = df.iloc[:-EVAL_POINTS], df.iloc[-EVAL_POINTS:]

model = Prophet(interval_width=INTERVAL_WIDTH, daily_seasonality=DAILY_SEASONALITY)
model.fit(train)

band = model.predict(tail[['ds']])[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
band['actual'] = tail['y'].values
band

## Visualize actual vs. predicted band

In [ ]:
import matplotlib.pyplot as plt

context = train.tail(EVAL_POINTS * 6)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(context['ds'], context['y'], color='#888', lw=1, label='history')
ax.fill_between(band['ds'], band['yhat_lower'], band['yhat_upper'],
                alpha=0.25, label=f'{int(INTERVAL_WIDTH * 100)}% band')
ax.plot(band['ds'], band['yhat'], '--', color='steelblue', label='yhat')
ax.plot(band['ds'], band['actual'], color='crimson', marker='o', ms=3, label='actual')
ax.set_title(f'Prophet forecast vs actual — {key[:60]}')
ax.legend(loc='upper left')
plt.tight_layout()

## Score the anomaly

For each scored point: `0.0` if it sits inside the band, otherwise the distance
outside as a fraction of the band width (capped at `1.0`). The series score is the
worst point in the window — the same logic the `anomaly-scorer` exports.

In [ ]:
def compute_anomaly_score(actual, lower, upper):
    scores = []
    for a, l, u in zip(actual, lower, upper):
        if l <= a <= u:
            scores.append(0.0)
        else:
            deviation = max(l - a, a - u)
            band_width = (u - l) if u != l else 1.0
            scores.append(min(deviation / band_width, 1.0))
    return max(scores) if scores else 0.0

score = compute_anomaly_score(band['actual'].values,
                              band['yhat_lower'].values,
                              band['yhat_upper'].values)
print(f'anomaly score for [{key}] = {score:.3f}')

## From notebook to exporter

The `anomaly-scorer` extra container (`extra/anomaly-scorer/`) runs this exact
fit -> predict -> score loop for **every** series the query returns and publishes
the result as a Prometheus gauge, so you can alert on it like any other metric:

```python
from prometheus_client import start_http_server, Gauge

ANOMALY = Gauge('custom_anomaly_score', 'Prophet anomaly score', ['namespace', 'pod'])
start_http_server(8000)
while True:
    for key, df in query_range(METRIC_QUERY).items():
        # ... fit / predict / score as above ...
        ANOMALY.labels(namespace=ns, pod=pod).set(score)
    time.sleep(300)
```

Scrape `:8000`, then alert with e.g. `custom_anomaly_score > 0.8`.

### Where to go next

- Promote the threshold to a mixin `prometheusRules` alert.
- Add `weekly_seasonality` or extra regressors for spikier signals.
- Swap the band score for a residual z-score if you want a smoother gradient.